# Dataset Exploration

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/content/PhiUSIIL_Phishing_URL_Dataset.csv')

In [3]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [4]:
df.shape

(235795, 56)

In [5]:
df['label'].value_counts()

,count
label,
1,134850
0,100945


# Dataset Split

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
df = df[['URL', 'label']]

In [8]:
df = df.dropna(subset=['label'])

In [9]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

In [10]:
test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=42, stratify=test_df['label'])

In [11]:
print(f"Training set: {len(train_df)}, Validation set: {len(val_df)}, Test set: {len(test_df)}")

Training set: 165056, Validation set: 35370, Test set: 35369


In [12]:
print("Training label distribution:\n", train_df['label'].value_counts())
print("Validation label distribution:\n", val_df['label'].value_counts())
print("Test label distribution:\n", test_df['label'].value_counts())

Training label distribution:
 label
1    94395
0    70661
Name: count, dtype: int64
Validation label distribution:
 label
1    20228
0    15142
Name: count, dtype: int64
Test label distribution:
 label
1    20227
0    15142
Name: count, dtype: int64


# Dataset Preprocessing

In [13]:
from transformers import BertTokenizer

In [14]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [15]:
def tokenize_function(examples):
  return tokenizer(examples['URL'], padding="max_length", truncation=True)

In [16]:
train_encodings = tokenizer(list(train_df['URL'].values), truncation=True, padding=True, max_length=512)

In [17]:
val_encodings =  tokenizer(list(val_df['URL'].values), truncation=True, padding=True, max_length=512)

In [18]:
test_encodings = tokenizer(list(test_df['URL'].values), truncation=True, padding=True, max_length=512)

# PyTorch Datasets

In [19]:
import torch

In [20]:
torch.cuda.empty_cache()

In [21]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [22]:
train_labels = train_df['label'].map({0: 0, 1: 1}).tolist()
val_labels = val_df['label'].map({0: 0, 1: 1}).tolist()
test_labels = test_df['label'].map({0: 0, 1: 1}).tolist()

In [23]:
train_dataset = TextDataset(train_encodings, train_labels)
val_dataset = TextDataset(val_encodings, val_labels)
test_dataset = TextDataset(test_encodings, test_labels)

# Model Setup

In [24]:
from transformers import AlbertForSequenceClassification, Trainer, TrainingArguments

In [25]:
model = AlbertForSequenceClassification.from_pretrained('albert-base-uncased', num_labels=2)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Training the Model

In [26]:
from sklearn.metrics import precision_recall_fscore_support

In [27]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none",
    fp16=True,
    logging_strategy="epoch",
    logging_steps=50,
    save_total_limit=1,
    save_steps=500,
    gradient_accumulation_steps = 2  # or higher

)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [28]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)  # Get the predicted class with the highest score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [30]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


# Evaluation

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

# Save Model

In [ ]:
save_directory = "/content/results_model"

In [ ]:
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

In [ ]:
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

In [ ]:
model.save_pretrained(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

In [ ]:
torch.save(model.state_dict(), './results/pytorch_model.bin')